# Credit Risk Project — Data Understanding

## Purpose of This Notebook

This notebook performs an initial exploration of a credit card default dataset to:

- Understand the structure and scale of the data
- Identify demographic, financial, and behavioral variables relevant to credit risk
- Examine the default target variable and its distribution
- Surface potential data quality issues and risk signals

This phase focuses strictly on **understanding the data**, without performing any cleaning, feature engineering, or modeling.

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

### Load the dataset

In [3]:
df=pd.read_excel("default of credit card clients.xls", header=1)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### Dataset Shape & Structure

In [4]:
df.shape

(30000, 25)

#### The dataset contains 30000 records with 25 columns.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

## Dataset Structure & Business Context

Each row in this dataset represents a **unique credit card customer**, observed over a six-month historical window.

The dataset combines:
- Customer demographics
- Credit exposure (approved limits)
- Historical repayment behavior
- Monthly billing and payment activity
- A binary default outcome for the subsequent month

This structure allows analysis of **who defaults**, **how they behave prior to default**, and **which behavioral patterns signal elevated credit risk**.



## Column Categories

The variables can be grouped based on their role in credit risk assessment.

### Customer Demographics
- SEX: Gender of the customer (encoded)
- EDUCATION: Education level (encoded categories)
- MARRIAGE: Marital status (encoded categories)
- AGE: Age of the customer

These variables provide baseline customer context and are typically secondary predictors.

### Credit Exposure
- LIMIT_BAL: Approved credit limit

This represents maximum potential exposure in the event of default.

Repayment History (Primary Risk Signals)
PAY_0 to PAY_6: Repayment status over the past six months

These variables encode delinquency severity and are among the strongest predictors of default in consumer credit risk.

They capture whether payments were made on time, delayed, or significantly overdue, directly reflecting behavioral credit risk.


### Billing Amounts
- BILL_AMT1 to BILL_AMT6: Monthly statement balances

These represent outstanding exposure carried month to month.

### Payment Amounts
- PAY_AMT1 to PAY_AMT6: Actual payments made each month

These reflect observed repayment behavior and liquidity.



### Target Variable Distribution

In [6]:
df['default payment next month'].value_counts(normalize=True)

,proportion
default payment next month,
0,0.7788
1,0.2212


### Target Variable
- default payment next month
  - 1 = Customer defaulted
  - 0 = Customer did not default

This is the outcome variable the risk model will predict.


## Target Variable Insight

The target variable is **imbalanced**, with the majority of customers not defaulting.

Business implications:
- Defaults are relatively rare but financially significant
- Accuracy alone is misleading
- Precision, recall, and cost-sensitive evaluation will be critical in modeling

This imbalance reflects real-world credit portfolios.

The target variable reflects a real-world credit portfolio where defaults are infrequent but financially material.


### Credit Limit & Age Sanity Checks

In [7]:
df[['LIMIT_BAL', 'AGE']].describe()

,LIMIT_BAL,AGE
count,30000.000000,30000.000000
mean,167484.322667,35.485500
std,129747.661567,9.217904
min,10000.000000,21.000000
25%,50000.000000,28.000000
50%,140000.000000,34.000000
75%,240000.000000,41.000000
max,1000000.000000,79.000000


In [8]:
df.isnull().sum().sort_values(ascending=False).head()

,0
ID,0
LIMIT_BAL,0
SEX,0
EDUCATION,0
MARRIAGE,0


##Risk & Data Quality Observations

In [11]:
df[['EDUCATION', 'MARRIAGE', 'SEX']].value_counts().head()

EDUCATION  MARRIAGE  SEX
2          1         2      4472
1          2         2      4176
2          2         2      4080
                     1      2940
1          2         1      2633
Name: count, dtype: int64

### Encoded Category Validation

Several categorical variables use numeric encodings.
Certain values (e.g., EDUCATION = 0 or values outside documented ranges) may represent unknown or miscoded categories and will require recoding during data preparation.


## Data Quality Notes

- No missing values detected across columns
- Several categorical variables are numerically encoded and require mapping
- Certain encoded values (e.g., EDUCATION = 0) likely represent unknown or undefined categories
- Repayment status values must be validated to ensure they fall within expected domains


### Repayment Status Variable Exploration

In [9]:
repayment_cols = [col for col in df.columns if 'PAY_' in col and col != 'PAY_AMT1']
df[repayment_cols].describe()

,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000
mean,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,-0.291100,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567
std,1.123802,1.197186,1.196868,1.169139,1.133187,1.149988,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775
min,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000
25%,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000
max,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000


## Repayment Status Overview

The PAY_* variables capture repayment behavior over the previous six months and encode delinquency severity.

Interpretation:
- Values of -2 indicate no credit consumption
- Values of -1 indicate payment made on time
- Value 0 indicates revolving credit
- Positive values (1–9) indicate delayed payments, with larger values representing longer delinquency periods

Business relevance:
- These variables are among the strongest predictors of credit default
- Persistent or increasing delays signal elevated default risk
- Repayment status reflects behavioral risk rather than income or balance size



## Key Takeaways from Data Understanding

- The dataset is complete, with no missing values
- Default events are relatively rare, indicating class imbalance
- Repayment status variables capture meaningful delinquency behavior
- Credit limits and billing amounts vary significantly, implying heterogeneous exposure
- Encoded categorical variables will require careful preprocessing

The next phase will focus on data cleaning, encoding validation, and feature engineering while preserving target integrity.
